In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/CV-2026-Project1.zip"



Mounted at /content/drive
mapname:  conversion of  failed


In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import math
from glob import glob



def align_using_sift(kp_empty_board, desc_empty_board, empty_board_shape, path_image):

    #read img
    image = cv2.imread(path_image)

    if image is None:
        raise ValueError(f"Could not read image: {path_image}")

    #grayscale for sift
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    #clahe for lighting/shadows normalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_img = clahe.apply(gray_img)

    #use sift
    sift = cv2.SIFT_create()

    #taking keypoints and descriptors
    kp_image, desc_image = sift.detectAndCompute(gray_img, None)

    #KD-Tree algorithm
    index_params = dict(algorithm=1, trees=5)

    #dictates how accurately it searches
    search_params = dict(checks=50)

    #using flann for matching descriptors between two images
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    #taking the 2 best matches for every desciptor
    matches = flann.knnMatch(desc_image, desc_empty_board, k=2)

    #lowes ratio test, filters bad matches, the 2 best matches need to not be similar, else ambiguous
    good_matches = [d for d, r in matches if d.distance < 0.7 * r.distance]

    #extract coord for best matches
    src_pts = np.float32([kp_image[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp_empty_board[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    #ransac for ignoring outliers
    #homography for making matrix that maps the image points to the empty board points
    H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    #height and width of empty board
    h, w = empty_board_shape[:2]

    #aligning image using H
    aligned_q = cv2.warpPerspective(image, H, (w, h))

    #from BGR to RGB
    return cv2.cvtColor(aligned_q, cv2.COLOR_BGR2RGB)



def hex_grid_sift(center_x, center_y, distance_x, distance_y):

    #center for center of board
    #distance for distance between centers of hexagons

    #nr of hexagons per row
    row_counts = [4, 7, 8, 9, 10, 9, 10, 9, 8, 7, 4]
    grid = {}
    cnt = 1


    for i, count in enumerate(row_counts):

        #row above middle row become negative, we start at center of game board
        row_offset = i - 5

        #y for row
        y = center_y + (row_offset * distance_y)

        #using x to align the hex rows
        start_x_offset = -(count - 1) / 2.0

        for j in range(count):

            #x for cell
            x = center_x + ((start_x_offset + j) * distance_x)

            #save coord
            grid[cnt] = (int(x), int(y))
            cnt += 1

    return grid



def detect_positions(aligned_prev, aligned_curr, grid_dict):

    #convert to grayscale
    gray_prev = cv2.cvtColor(aligned_prev, cv2.COLOR_RGB2GRAY)
    gray_curr = cv2.cvtColor(aligned_curr, cv2.COLOR_RGB2GRAY)

    #gaussian blur for smooth image
    gray_prev = cv2.GaussianBlur(gray_prev, (5, 5), 0)
    gray_curr = cv2.GaussianBlur(gray_curr, (5, 5), 0)

    #radius to take entire hexagon
    cell_drops = {}
    radius = 60


    for idx, (cx, cy) in grid_dict.items():

        #makes a black image
        mask = np.zeros_like(gray_prev)
        #draws a white circle on black image at our target location
        cv2.circle(mask, (cx, cy), radius, 255, -1)


        #takes grayscale pixels from inside white circle
        pixels_prev = gray_prev[mask == 255]
        pixels_curr = gray_curr[mask == 255]

        #calculate the average brightness of hexagon and the difference between the 2
        mean_prev = np.mean(pixels_prev)
        mean_curr = np.mean(pixels_curr)

        #brightness should drop if there is a new hexagon
        drop = mean_prev - mean_curr

        #if a tile was already there
        if mean_prev < 60:
             drop = -999

        cell_drops[idx] = drop

    #biggest brightness drop to smallest
    sorted_cells = sorted(cell_drops.items(), key=lambda item: item[1], reverse=True)

    #takes board position of biggest 2
    predicted_pair = [sorted_cells[0][0], sorted_cells[1][0]]


    return predicted_pair


def detect_colors_hsv(aligned_curr, predicted_positions, grid_dict):

    #convert to HSV
    hsv_img = cv2.cvtColor(aligned_curr, cv2.COLOR_RGB2HSV)

    #split channels
    h, s, v = cv2.split(hsv_img)

    #clahe for lighting/shadows normalization on value channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    new_v = clahe.apply(v)

    #merge back
    hsv_img = cv2.merge((h, s, new_v))

    #radius of what we are looking at
    radius = 20
    predicted_colors = {}


    for pos in predicted_positions:

        #pixel coord
        x_pos, y_pos = grid_dict[pos]

        #manual shifting
        y_pos = y_pos - 30

        #makes a black image
        mask = np.zeros(hsv_img.shape[:2], dtype=np.uint8)

        #draws a white circle on black image at our target location
        cv2.circle(mask, (x_pos, y_pos), radius, 255, -1)

        #takes hsv pixels from inside white circle
        hsv_pixels = hsv_img[mask == 255]

        color_counts = {'R': 0, 'G': 0, 'B': 0, 'Y': 0, 'O': 0, 'P': 0}

        #using hue as color classf of a pixel
        for pixel in hsv_pixels:

            hue, s, v = pixel

            #hue from 0 to 180
            if hue <= 8 or hue >= 165:
                color_counts['R'] += 1
            elif 9 <= hue <= 24:
                color_counts['O'] += 1
            elif 22 <= hue <= 38:
                color_counts['Y'] += 1
            elif 39 <= hue <= 75:
                color_counts['G'] += 1
            elif 76 <= hue <= 105:
                color_counts['B'] += 1
            elif 106 <= hue <= 164:
                color_counts['P'] += 1

        #choosing highest score
        best_color = max(color_counts, key=color_counts.get)

        #anti white glare measure
        if color_counts[best_color] == 0:
            best_color = 'R'

        predicted_colors[pos] = best_color


    return predicted_colors


def making_coordinates():

    #nr of hexagons per row
    row_counts = [4, 7, 8, 9, 10, 9, 10, 9, 8, 7, 4]
    coords = {}
    hex = 1


    for row_idx, count in enumerate(row_counts):

        #using x to align the hex rows
        start_x = -(count - 1) / 2.0

        #calculating final x and y
        for col_idx in range(count):
             coords[hex] = (start_x + col_idx, row_idx)
             hex += 1

    #manually mapping the 6 hexagons from the beggining of the game
    coords['R_sym'] = (-2.5, 0)
    coords['G_sym'] = (2.5, 0)
    coords['P_sym'] = (-5.0, 5)
    coords['B_sym'] = (5.0, 5)
    coords['Y_sym'] = (-2.5, 10)
    coords['O_sym'] = (2.5, 10)

    return coords



def get_neighbors(cell, coords):

    #if hexagon doesnt exist
    if cell not in coords:
        return []

    cell_x, cell_y = coords[cell]

    #the 6 directions of coord: right, left, bottom right, bottom left, top right, top left
    directions = [
        (1, 0),
        (-1, 0),
        (0.5, 1),
        (-0.5, 1),
        (0.5, -1),
        (-0.5, -1)
    ]

    neighbors = []
    #check the 6 directions
    for n_idx, (nx, ny) in coords.items():

        if n_idx == cell:
            continue

        for dx, dy in directions:
            #check if its a neighbor and if yes, get its id and direction
            if math.isclose(cell_x + dx, nx, abs_tol=0.01) and math.isclose(cell_y + dy, ny, abs_tol=0.01):
                neighbors.append((n_idx, dx, dy))

    return neighbors



def calculate_move_score(predicted_positions, predicted_colors, current_board_state, coords):
    move_score = {}


    #we take each of the 2 hexagons
    for start_pos in predicted_positions:


        color = predicted_colors[start_pos]
        #initialize color
        if color not in move_score:
            move_score[color] = 0

        neighbors = get_neighbors(start_pos, coords)


        for n_idx, dx, dy in neighbors:

            #skip if the hexagon is the other half of placed tile
            if n_idx in predicted_positions:
                continue

            #following the line in the current direction
            current_id = n_idx
            current_x, current_y = coords[n_idx]

            #walk forward if there is a hexagon and the same color
            while current_id in current_board_state and current_board_state[current_id] == color:
                move_score[color] += 1

                #step to the next hexagon in the same direction
                next_x = current_x + dx
                next_y = current_y + dy
                next_id = None

                #find next hexagon
                for k, (x, y) in coords.items():
                    if math.isclose(x, next_x, abs_tol=0.01) and math.isclose(y, next_y, abs_tol=0.01):
                        next_id = k
                        break

                #reached the edge of the board
                if next_id is None:
                    break

                current_id = next_id
                current_x, current_y = coords[next_id]

    #remove colors with 0 score
    return {c: v for c, v in move_score.items() if v > 0}


def detected_table_score(table_scores_dir):

    database = {}

    #hsv color boundaries for the 6 scoring cubes, need 2 for red bcs of color wheel
    color_bounds = {
        'R': ([0, 150, 100], [10, 255, 255]),
        'R2': ([170, 150, 100], [180, 255, 255]),
        'G': ([35, 50, 40], [85, 255, 255]),
        'O': ([5, 100, 150], [22, 255, 255]),
        'B': ([95, 150, 120], [130, 255, 255]),
        'Y': ([23, 100, 150], [35, 255, 255]),
        'P': ([120, 40, 40], [165, 255, 255])
    }

    #gathers every scorecard image
    image_paths = glob(os.path.join(table_scores_dir, "*.jpg"))


    for count, img_path in enumerate(image_paths):

        print(f"image {count+1}, {img_path}")
        filename = os.path.basename(img_path)
        img = cv2.imread(img_path)

        if img is None: continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)

        #using canny edge detection, it only looks for sharp changes in contrast, ignores shadows/glare
        edges = cv2.Canny(blurred, 50, 150)

        #thicken edges so that board outline is closed
        kernel = np.ones((5,5), np.uint8)
        dilated_edges = cv2.dilate(edges, kernel, iterations=2)

        #find contours, closed areas
        contours, _ = cv2.findContours(dilated_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        y_min_score = 650
        y_max_score = 50

        if contours:
            #keep the largest contour
            board_contour = max(contours, key=cv2.contourArea)

            if cv2.contourArea(board_contour) > 10000:

                #bounding box coord of board
                board_x, board_y, board_w, board_h = cv2.boundingRect(board_contour)

                #calculating where the "18" line is from boards total height + manual tuning
                y_max_score = int(board_y + (board_h * 0.07)) -80

                #calculating where the "0" line is from boards total height + manual tuning
                y_min_score = int(board_y + (board_h * 0.83)) +220

            else:
                print("Using defaults")
        else:
            print("Using defaults")


        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        img_score = {'R': 0, 'G': 0, 'O': 0, 'B': 0, 'Y': 0, 'P': 0}

        #loop through each color
        for color, name in zip(['R', 'G', 'O', 'B', 'Y', 'P'], ['Red', 'Green', 'Orange', 'Blue', 'Yellow', 'Purple']):

            #creating a mask that isolates only pixels matching color
            lower, upper = color_bounds[color]
            mask = cv2.inRange(hsv, np.array(lower), np.array(upper))

            #same but for the case of red second range
            if color == 'R':
                l2, u2 = color_bounds['R2']
                mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.array(l2), np.array(u2)))

            #find contours, closed areas for color
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if contours:

                #take the biggest color blob
                max_c = max(contours, key=cv2.contourArea)

                if cv2.contourArea(max_c) > 200:
                    x, y, w, h = cv2.boundingRect(max_c)
                    center_y = int(y + (h / 2))

                    #the total distance between line 0 and 18
                    span = y_min_score - y_max_score

                    if span != 0:

                        #distance between line 0 and pixel center, getting percentage of the way up the board, then score
                        raw_score = ((y_min_score - center_y) / span) * 18

                        #rounded to the nearest whole number, and forced between 0 and 18
                        final_score = max(0, min(18, int(round(raw_score))))

                        img_score[color] = final_score

                else:
                    print("not found")
            else:
                 print("not found")

        database[filename] = img_score

    return database



def save_outputs(move_str, positions, colors, score, predicted_table_img, mapping_file_path, output_dir="predictions"):

    #making folder for output
    os.makedirs(output_dir, exist_ok=True)

    #create txt move file
    move_file_path = os.path.join(output_dir, f"{move_str}.txt")

    with open(move_file_path, 'w') as f:
        #sort positions
        point1, point2 = sorted(positions)

        f.write(f"{point1} {colors[point1]}\n")
        f.write(f"{point2} {colors[point2]}\n")


        if score:
            #order for score if score is the same
            color_order = ['R', 'G', 'O', 'B', 'Y', 'P']
            score_parts = []

            for color in color_order:

                if color in score and score[color] > 0:
                    score_parts.append(f"{score[color]}{color}")

            score_str = " ".join(score_parts)
            f.write(f"{score_str}\n")
        else:
            f.write("\n")

    #updating mapping txt file
    with open(mapping_file_path, 'a') as f:
        f.write(f"{move_str}.jpg {predicted_table_img}\n")



images_dir = "C:/Users/domin/Downloads/CV-2026-Project1/train/boards"
empty_board_path = 'board_1.jpg'
table_scores_dir = "C:/Users/domin/Downloads/CV-2026-Project1/train/table-scores"

#ensure output directory exists
PREDICTIONS_DIR = './predictions'
os.makedirs(PREDICTIONS_DIR, exist_ok=True)



#making scores from table scores
table_scores = detected_table_score(table_scores_dir)

#prepare empty board game for sift
board_img = cv2.imread(empty_board_path)
board_img_rgb = cv2.cvtColor(board_img, cv2.COLOR_BGR2RGB)
gray_board = cv2.cvtColor(board_img, cv2.COLOR_BGR2GRAY)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
gray_board = clahe.apply(gray_board)

#use sift
sift = cv2.SIFT_create()
kp_board, des_board = sift.detectAndCompute(gray_board, None)

#manual coordinates for center of board game and spacing between hexagons
CENTER_X = 2055
CENTER_Y = 1460
DX = 144
DY = 124

#making pixel map for position/color and math map for scoring
grid_dict = hex_grid_sift(CENTER_X, CENTER_Y, DX, DY)
hex_coords = making_coordinates()


#starting states for the 6 colored hexagons
INITIAL_BOARD_STATE = {
    'R_sym': 'R', 'G_sym': 'G',
    'P_sym': 'P', 'B_sym': 'B',
    'Y_sym': 'Y', 'O_sym': 'O'
}

processed_images = 0

for g in range(1, 6):

    #start each game with empty board
    aligned_prev = board_img_rgb

    #initialize the board state for a new game
    current_board_state = dict(INITIAL_BOARD_STATE)


    #cumulative scores for both players at the start of the game
    player_scores = {
        1: {'R': 0, 'G': 0, 'O': 0, 'B': 0, 'Y': 0, 'P': 0},
        2: {'R': 0, 'G': 0, 'O': 0, 'B': 0, 'Y': 0, 'P': 0}
    }

    #create mapping file for game
    mapping_file_path = os.path.join(PREDICTIONS_DIR, f"{g}_table_scores_mapping.txt")
    open(mapping_file_path, 'w').close()



    for m in range(1, 21):
        move_str = f"{g}_{m:02d}"
        curr_img_path = os.path.join(images_dir, f"{move_str}.jpg")
        annotation_path = os.path.join(images_dir, f"{move_str}.txt")

        if not os.path.exists(curr_img_path):
            continue

        processed_images = processed_images + 1
        print(f"Processing {move_str}")

        #determines active player, player 1 odd moves, player 2 even moves
        active_player = 1 if m % 2 != 0 else 2

        try:
            #align image to empty board
            aligned_curr = align_using_sift(kp_board, des_board, board_img.shape, curr_img_path)

            #find positions of new hexagons
            predicted_positions = detect_positions(aligned_prev, aligned_curr, grid_dict)

            #predict colors
            predicted_colors = detect_colors_hsv(aligned_curr, predicted_positions, grid_dict)

            #calculate score
            calculated_score = calculate_move_score(predicted_positions, predicted_colors, current_board_state, hex_coords)

            #update cumulative score for active player
            for color, pts in calculated_score.items():
                 #maximum board value of 18
                 player_scores[active_player][color] = min(18, player_scores[active_player][color] + pts)

            #finding matching table image
            current_target_score = player_scores[active_player]
            predicted_table_img = None

            for img_name, img_vals in table_scores.items():
                if img_vals == current_target_score:
                     predicted_table_img = img_name
                     break

            if predicted_table_img is None:
                 predicted_table_img = "None Found"


            save_outputs(move_str, predicted_positions, predicted_colors, calculated_score, predicted_table_img, mapping_file_path, output_dir=PREDICTIONS_DIR)

            #update the board state info
            for pos in predicted_positions:
                 current_board_state[pos] = predicted_colors[pos]

            aligned_prev = aligned_curr

        except Exception as e:
            print(f"Error processing {move_str}: {e}")


